# Medical Reasoning LLM Fine-tuning with Unsloth

This notebook fine-tunes a small language model for medical reasoning using **Unsloth**:
- **2-5x faster training** than standard methods
- **50% less memory usage**
- **LoRA + 4-bit quantization** built-in
- **Medical O1 Reasoning Dataset** for training
- **Optimized for Google Colab GPU**
- **No complex setup required**

## 1. Install Unsloth

In [ ]:
# Install Unsloth with all dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install additional dependencies
!pip install --no-deps trl peft accelerate bitsandbytes

print("Unsloth installed successfully!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-b1nlllfo/unsloth_818d2a40eb1a4469933e40c2a1ad3f06
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-b1nlllfo/unsloth_818d2a40eb1a4469933e40c2a1ad3f06
  Resolved https://github.com/unslothai/unsloth.git to commit f06200db07e4df3e011afe9dbe671c55348d4d9b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.1/206.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 10.6 MB/s eta 0:

## 2. Import Libraries and Setup

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
import torch
import json
from typing import Dict, List, Any
import os

# Disable wandb
os.environ["WANDB_MODE"] = "disabled"

# Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA available: True
GPU: Tesla T4
Memory: 14.7 GB


## 3. Load Pre-trained Model

In [ ]:
# Load model - Unsloth supports many models out of the box
# Choose from: llama, mistral, gemma, qwen, phi, etc.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-2-7b-bnb-4bit",  # Fast 4bit Llama-2 7B
    # Other options:
    # "unsloth/mistral-7b-bnb-4bit",
    # "unsloth/phi-3-mini-4k-instruct-bnb-4bit",
    # "unsloth/gemma-7b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded: {model.config.name_or_path}")
print(f"Model parameters: {model.num_parameters():,}")

==((====))==  Unsloth 2025.9.4: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.87G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: unsloth/llama-2-7b-bnb-4bit
Model parameters: 6,738,415,616


## 4. Add LoRA Adapters

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16, # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0, # Supports any, but = 0 is optimized
    bias="none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth", # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None, # And LoftQ
)

# Show trainable parameters
model.print_trainable_parameters()

Unsloth 2025.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 39,976,960 || all params: 6,778,392,576 || trainable%: 0.5898


## 5. Load and Format Dataset

In [ ]:
def load_medical_dataset():
    """Load the medical reasoning dataset from local JSONL files"""

    def load_jsonl(file_path):
        data = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line.strip()))
        return data

    try:
        # Load data from JSONL files
        train_data = load_jsonl('/content/drive/MyDrive/MSc/Medical-O1-Dataset/medical_o1_train.jsonl')
        val_data = load_jsonl('/content/drive/MyDrive/MSc/Medical-O1-Dataset/medical_o1_validation.jsonl')

        print(f"Loaded {len(train_data)} training examples")
        print(f"Loaded {len(val_data)} validation examples")

        return train_data, val_data

    except FileNotFoundError:
        # Fallback: load from HuggingFace
        print("Local files not found, loading from HuggingFace...")
        from datasets import load_dataset
        dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")

        train_data = [ex for ex in dataset['train']]
        val_data = [ex for ex in dataset['validation']]

        return train_data, val_data

# Load the data
train_data, val_data = load_medical_dataset()

Loaded 13792 training examples
Loaded 1972 validation examples


In [ ]:
# Alpaca prompt format for medical reasoning
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    """Format the medical conversations for training"""
    instructions = []
    inputs = []
    outputs = []
    texts = []

    for messages in examples["messages"]:  # Fixed: messages is already the list
        # Extract system message as instruction
        instruction = "You are a medical AI assistant trained to provide detailed medical reasoning and analysis."

        # Extract user question and assistant response
        user_message = ""
        assistant_message = ""

        for msg in messages:  # Fixed: iterate directly over messages
            if msg["role"].lower() in ["user"]:
                user_message = msg["content"]
            elif msg["role"].lower() in ["chatbot", "assistant"]:
                assistant_message = msg["content"]

        # Format using Alpaca template
        text = alpaca_prompt.format(instruction, user_message, assistant_message) + EOS_TOKEN
        texts.append(text)

        instructions.append(instruction)
        inputs.append(user_message)
        outputs.append(assistant_message)

    return {
        "text": texts,
        "instruction": instructions,
        "input": inputs,
        "output": outputs,
    }


# Limit dataset size for faster training in Colab
train_subset = train_data[:1000]  # Use 1000 samples
val_subset = val_data[:200]       # Use 200 samples

# Create datasets - Fixed the structure
train_dataset = Dataset.from_list([{"messages": ex["messages"]} for ex in train_subset])
val_dataset = Dataset.from_list([{"messages": ex["messages"]} for ex in val_subset])

# Apply formatting
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

# Show example
print("\nExample formatted prompt:")
print(train_dataset[0]["text"][:500] + "...")


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Training dataset size: 1000
Validation dataset size: 200

Example formatted prompt:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a medical AI assistant trained to provide detailed medical reasoning and analysis.

### Input:
A 72-year-old man presents with symptoms that include a cough, painful breathing, weight loss, swelling of the face and arms, and distended veins in the chest and arms, along with right eye ptosis and miosis. Given these ...


## 6. Start Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling, EarlyStoppingCallback
import torch

# Calculate steps for 10 epochs (info only)
total_train_samples = len(train_dataset)
batch_size = 2
gradient_accumulation_steps = 4
effective_batch_size = batch_size * gradient_accumulation_steps
steps_per_epoch = total_train_samples // effective_batch_size
total_steps = steps_per_epoch * 4

print(f"Training setup:")
print(f"Total samples: {total_train_samples}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Steps/epoch: {steps_per_epoch}")
print(f"Total steps if 10 epochs: {total_steps}")

# Fix pad token
tokenizer.pad_token = tokenizer.eos_token

# Use data collator with static shapes
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

args = TrainingArguments(
    output_dir="outputs",
    seed=3407,

    # Batching
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    # Training schedule
    num_train_epochs=4,              # shorter run
    learning_rate=5e-5,
    weight_decay=0.05,               # stronger regularization
    lr_scheduler_type="cosine",
    warmup_steps=int(total_steps * 0.1),

    # Precision
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    # Logging / eval / saving
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=steps_per_epoch,
    save_strategy="steps",
    save_steps=steps_per_epoch,
    save_total_limit=5,

    # Best checkpoint handling
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to=None,
    optim="adamw_8bit"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=args,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001
        )
    ]
)

Training setup:
Total samples: 1000
Effective batch size: 8
Steps/epoch: 125
Total steps if 10 epochs: 500


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
# Show memory usage before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"Memory before training = {start_gpu_memory} GB.")

# Start training
trainer_stats = trainer.train()

# Show memory usage after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"Peak reserved memory = {used_memory} GB ({used_percentage}%).")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB ({lora_percentage}%).")
print(f"Peak reserved memory % of {max_memory} GB = {used_percentage}%.")

print("\nTraining completed! 🚀")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


GPU = Tesla T4. Max memory = 14.741 GB.
Memory before training = 6.369 GB.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 4 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
125,0.913600,0.907310
250,0.896800,0.884242
375,0.832500,0.880923
500,0.786800,0.883706


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Peak reserved memory = 6.369 GB (43.206%).
Peak reserved memory for training = 0.0 GB (0.0%).
Peak reserved memory % of 14.741 GB = 43.206%.

Training completed! 🚀


## 7. Save the Model

In [ ]:
# Save the LoRA adapters
model.save_pretrained("medical_reasoning_lora")
tokenizer.save_pretrained("medical_reasoning_lora")

print("LoRA adapters saved to 'medical_reasoning_lora' folder")

# Save as merged model (optional - takes more space but easier to use)
if False:  # Set to True if you want to save merged model
    model.save_pretrained_merged("medical_reasoning_merged", tokenizer, save_method="merged_16bit")
    print("Merged model saved to 'medical_reasoning_merged' folder")

LoRA adapters saved to 'medical_reasoning_lora' folder


In [ ]:
# Copy saved model to Google Drive
!cp -r medical_reasoning_lora /content/drive/MyDrive/MSc/Medical-Reasoning-LLM/

## 8. Test the Fine-tuned Model

In [ ]:
# Enable fast inference
FastLanguageModel.for_inference(model)

def ask_medical_question(question: str):
    """Ask the fine-tuned model a medical question"""
    prompt = alpaca_prompt.format(
        "You are a medical AI assistant trained to provide detailed medical reasoning and analysis.",
        question,
        "" # Leave response empty for generation
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
    )

    response = tokenizer.batch_decode(outputs)[0]

    # Extract just the response part
    if "### Response:" in response:
        response = response.split("### Response:")[1].strip()
        if tokenizer.eos_token in response:
            response = response.split(tokenizer.eos_token)[0].strip()

    return response

# Test questions
test_questions = [
    "What are the symptoms of pneumonia?",
    "A 45-year-old patient presents with chest pain and shortness of breath. What should be the first diagnostic step?",
    "Explain the mechanism of action of ACE inhibitors in treating hypertension.",
    "What is the difference between Type 1 and Type 2 diabetes?"
]

print("Testing the fine-tuned medical reasoning model:\n")

for i, question in enumerate(test_questions, 1):
    print(f"🏥 Q{i}: {question}")
    response = ask_medical_question(question)
    print(f"🤖 A{i}: {response}")
    print("-" * 80)


Testing the fine-tuned medical reasoning model:

🏥 Q1: What are the symptoms of pneumonia?
🤖 A1: The symptoms of pneumonia can vary widely depending on the type of pneumonia, the severity, and the individual's overall health. Common symptoms include:

1. **Cough**: A cough is often the first noticeable sign, and it may be dry or productive, meaning it produces phlegm.
2. **Fever**: Fever is a common symptom, as the body's immune response to the infection can lead to increased body temperature.
3. **Fatigue**: Pneumonia can significantly drain energy, leading to fatigue.
4. **Chills**: Shivering, also known as chills, is a common response to infection.
5. **Sharp or Dull Chest Pain**: Some types of pneumonia, such as lobar pneumonia, can cause sharp or dull chest pain.
6. **Shortness of Breath**: Shortness of breath, also known as dyspnea, can occur as the lungs become congested with fluid or inflammation.
7. **Sputum Production**: Pneumonia can cause increased production of s
---------

## 8. Load Model for Inference (Later Use)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="medical_reasoning_lora",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

print("Model loaded for inference!")